# <font color="#003660">Applied Machine Learning for Text Analysis (M.184.5331)</font>

# <font color="#003660">Session 5: Retrieval-Augmented Generation</font>

# <font color="#003660">Reranking</font>

<center><br><img width=256 src="https://raw.githubusercontent.com/olivermueller/aml4ta-2021/main/resources/dag.png"/><br></center>

<p>

<div>
    <font color="#085986"><b>By the end of this lesson, you ...</b><br><br>
        ... know how to implement reranking methods with LangChain, <br>
        ... know how to implement LangChain chains on your own.
    </font>
</div>
</p>

Inspired by these excellent sources:

* [HuggingFace (2024): NLP Course](https://huggingface.co/learn/nlp-course/)
* [Huggingface (2024): Open-Source AI Cookbook](https://huggingface.co/learn/cookbook/index)
* [LangChain Docs: Build a RAG agent](https://docs.langchain.com/oss/python/langchain/rag)
* [LangChain API Reference](https://reference.langchain.com/python/)
* [LangChain AI (2024) Cookbook](https://github.com/langchain-ai/langchain/blob/master/cookbook/rewrite.ipynb?ref=blog.langchain.dev)

## Where We Are

* **5.03:** RAG with LangChain. The retriever returns the 3 chunks that are most similar to the question.
* **Now:** **reranking**. Retrieve more chunks, then let a second model pick the best ones.

![RAG extensions](https://raw.githubusercontent.com/olivermueller/amlta-2026/main/Session_05/imgs/rag_extensions.png)

(Source: [Wang et al., 2024](https://doi.org/10.18653/v1/2024.emnlp-main.981))

Reranking is one of many ways to improve RAG ([Wang et al., 2024](https://doi.org/10.18653/v1/2024.emnlp-main.981)).

In [ ]:
%pip install -q langchain langchain-classic langchain-openai langchain-huggingface langchain-community langchain-text-splitters faiss-cpu sentence-transformers flashrank python-dotenv

In [ ]:
import os
import re
import urllib.request

from IPython.display import display, Markdown

from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_openai import ChatOpenAI
from langchain_huggingface import HuggingFaceEmbeddings
from transformers import AutoTokenizer
from langchain_community.vectorstores import FAISS
from langchain_core.prompts import PromptTemplate

from langchain_classic.retrievers import ContextualCompressionRetriever
from langchain_classic.retrievers.document_compressors import CrossEncoderReranker
from langchain_community.cross_encoders import HuggingFaceCrossEncoder
from langchain_community.document_compressors import FlashrankRerank

In [ ]:
# downloads the documents, if they are not here yet
REPO_URL = "https://raw.githubusercontent.com/olivermueller/amlta-2026/main/Session_05/"
for path in [
    "documents/Game_of_Thrones.pdf",
    "documents/How_I_Met_Your_Mother.pdf",
    "markdown_documents/Game_of_Thrones.md",
    "markdown_documents/How_I_Met_Your_Mother.md",
]:
    if not os.path.exists(path):
        os.makedirs(os.path.dirname(path), exist_ok=True)
        urllib.request.urlretrieve(REPO_URL + path, path)

### Choose where the model runs

We use two models:

* a **chat model** that writes the answers (the generator). It runs where you choose below.
* an **embedding model** that turns texts into vectors (the retriever). It always runs on your computer, see below.

| `provider` | Where the model runs | `model_name` | Good to know |
|---|---|---|---|
| `"ollama"` | **Ollama** on your computer | `qwen3.5:9b`, `qwen3.5:4b`, `qwen3.5:2b` | free, no account; install [Ollama](https://ollama.com/download) first, then download the model once in a terminal: `ollama pull qwen3.5:9b` (6.6 GB; 4b: 3.3 GB; 2b: 2.7 GB); without a GPU, it runs on the CPU, which is slower |
| `"colab"` | **Ollama on Google Colab** | the same three | free, needs a Google account; the cell installs Ollama in the Colab runtime and downloads the model, as in 4.01; choose a GPU runtime first (Runtime > Change runtime type > T4 GPU) |
| `"huggingface"` | **Hugging Face endpoints** of the course | the same three, one endpoint each | key and addresses from the course; if an endpoint was asleep, the first request takes 1 to 2 minutes |
| `"ai-gateway"` | **AI-Gateway** of Paderborn University | `gwdg.qwen3.8-27b-no-think` (thinking off), `gwdg.qwen3.8-27b` (thinking on) | your university key (see [setup/SETUP_AI_GATEWAY.md](https://github.com/olivermueller/amlta-2026/blob/main/setup/SETUP_AI_GATEWAY.md)); only in the university network or the VPN, not on Colab; about 10 requests per minute |
| `"other"` | **another** OpenAI-compatible API | the name your provider uses | base URL and key in `.env` (`OTHER_BASE`, `OTHER_KEY`) |

Keys and addresses come from the `.env` file in the repo folder (see `.env.example`). Without a `.env` file, for example on Colab, the cell asks for them.

In [ ]:
# Run this cell: it only defines setup_model(). We call it before every ChatOpenAI, with the provider and the model.
import getpass
import json
import os
import shutil
import subprocess
import sys
import time
import urllib.error
import urllib.request

from dotenv import load_dotenv

load_dotenv()  # reads the .env file if there is one: this folder first, then the folders above (the repo folder)
IN_COLAB = "google.colab" in sys.modules

PROVIDERS = ["ollama", "colab", "huggingface", "ai-gateway", "other"]
QWEN = {  # model_name: (download size in Ollama, the name of its endpoint address in .env, the model on Hugging Face)
    "qwen3.5:9b": ("6.6 GB", "HF_GATEWAY_BASE_9B", "unsloth/Qwen3.5-9B-GGUF"),
    "qwen3.5:4b": ("3.3 GB", "HF_GATEWAY_BASE_4B", "unsloth/Qwen3.5-4B-GGUF"),
    "qwen3.5:2b": ("2.7 GB", "HF_GATEWAY_BASE_2B", "unsloth/Qwen3.5-2B-GGUF"),
}
GATEWAY_QWEN = {  # on the AI-Gateway, the model name decides about thinking
    "gwdg.qwen3.8-27b-no-think": "thinking off",
    "gwdg.qwen3.8-27b": "thinking on",
}


def value(env_name, question, secret=False):
    """Takes the value from .env, or asks for it (keys are typed hidden)."""
    found = os.getenv(env_name)
    if found:
        return found.strip()
    answer = getpass.getpass(question).strip() if secret else input(question).strip()
    os.environ[env_name] = answer  # asked once: the next call of setup_model() takes it from here
    return answer


def start_ollama(model, size, colab=False):
    """Makes sure that the Ollama server runs and has the model. Downloads the model only on Colab."""
    url = (os.getenv("OLLAMA_BASE") or "http://localhost:11434").rstrip("/")

    def running():
        try:
            urllib.request.urlopen(url, timeout=2)
            return True
        except OSError:
            return False

    if colab and shutil.which("ollama") is None:
        # the same steps as in 4.01: the installer needs pciutils and zstd
        print("Installing Ollama in the Colab runtime (about a minute) ...")
        subprocess.run("sudo apt-get install -y -qq pciutils zstd > /dev/null && curl -fsSL https://ollama.com/install.sh | sh > /dev/null",
                       shell=True, check=True)
    if not running():
        if shutil.which("ollama") is None:
            raise RuntimeError("Ollama is not installed. Install it from https://ollama.com/download and run this cell again.")
        # on Colab, the installer may say that the server is running, but it is not; so we start it ourselves (as in 4.01)
        print("Starting the Ollama server ...")
        ollama_log = open("ollama.log", "w")  # a pipe that nobody reads could fill up and block the server
        subprocess.Popen(["ollama", "serve"], stdout=ollama_log, stderr=ollama_log)
        for _ in range(30):
            if running():
                break
            time.sleep(1)
        else:
            raise RuntimeError("The Ollama server did not start. Look into ollama.log.")

    installed = [m["name"] for m in json.load(urllib.request.urlopen(url + "/api/tags"))["models"]]
    if model not in installed and not colab:  # on your computer, you download the model yourself, once
        raise RuntimeError(f"{model} is not downloaded yet. Run this command in a terminal (it downloads {size}), "
                           f"then run this cell again:\n\n    ollama pull {model}\n")
    if model not in installed:
        print(f"WARNING: {model} is not here yet. It is downloaded now ({size}), this can take several minutes.")
        request = urllib.request.Request(url + "/api/pull", data=json.dumps({"model": model}).encode())
        shown = -1
        for line in urllib.request.urlopen(request):
            status = json.loads(line)
            if "error" in status:
                raise RuntimeError(status["error"])
            if status.get("total"):
                percent = int(100 * status.get("completed", 0) / status["total"]) // 10 * 10
                if percent != shown:
                    print(f"  {percent} %", end="  ")
                    shown = percent
        print("\nDownload finished.")
    return url + "/v1/"


def wait_until_awake(base_url, api_key):
    """A sleeping Hugging Face endpoint answers 503 until it has started; we wait for it."""
    request = urllib.request.Request(base_url + "/models", headers={"Authorization": f"Bearer {api_key}"})
    for attempt in range(40):  # up to about 6 minutes
        try:
            urllib.request.urlopen(request, timeout=10)
            return
        except urllib.error.HTTPError as error:
            if error.code == 403:
                raise RuntimeError("The endpoint answers 403: it is paused (the course endpoints only run during the seminar), "
                                   "or your key has no access to it. Choose another provider, or check HF_GATEWAY_KEY and the address in .env.")
            if error.code != 503:
                raise RuntimeError(f"The endpoint answers {error.code}. Check the key and the address.")
        except OSError:
            pass
        if attempt == 0:
            print("The endpoint is asleep and starts now. This takes a few minutes ...")
        time.sleep(8)
    raise RuntimeError("The endpoint did not wake up. Try again in a few minutes.")


def setup_model(model):
    """Checks {"model_name": ..., "provider": ...}, sets the address and the key, and prints what to keep in mind.
    Returns base_url, api_key and the model name for ChatOpenAI."""
    name = model.get("model_name", "").strip()
    provider = model.get("provider", "").strip().lower()
    if provider not in PROVIDERS:
        raise ValueError(f"Unknown provider {provider!r}. Choose one of: {', '.join(PROVIDERS)}.")
    if provider in ("ollama", "colab", "huggingface") and name not in QWEN:
        raise ValueError(f"The provider {provider!r} has these models: {', '.join(QWEN)}. Not {name!r}.")
    notes = []

    if provider in ("ollama", "colab"):
        if provider == "ollama" and IN_COLAB:
            raise ValueError("This notebook runs on Google Colab: use the provider 'colab'.")
        if provider == "colab" and not IN_COLAB:
            raise ValueError("The provider 'colab' only works on Google Colab. On your own computer, use 'ollama'.")
        if provider == "colab" and subprocess.run("nvidia-smi", shell=True, capture_output=True).returncode != 0:
            notes.append("This Colab runtime has no GPU: answers take long. Better: Runtime > Change runtime type > T4 GPU, then run the notebook again.")
        if provider == "ollama":
            notes.append("The model runs on this computer. Without a GPU, it runs on the CPU, and answers take longer.")
        base_url, api_key = start_ollama(name, QWEN[name][0], colab=(provider == "colab")), "ollama"  # Ollama needs no key
        notes.append('Thinking: switch it with "reasoning_effort" in the extra_body of ChatOpenAI ("none" = off).')
        notes.append("Several answers per request (n > 1): Ollama returns only one.")

    elif provider == "huggingface":
        _, env_name, name = QWEN[name]  # name: the model on Hugging Face
        base_url = value(env_name, f"Paste the address of the Hugging Face endpoint for {model['model_name']}: ").rstrip("/")
        if "<" in base_url:
            raise ValueError(f"The address for {model['model_name']} is still a placeholder in .env ({env_name}). "
                             "Paste the address from the course there, or choose another model or provider.")
        if not base_url.endswith("/v1"):
            base_url += "/v1"  # the OpenAI-compatible API of the endpoint
        api_key = value("HF_GATEWAY_KEY", "Paste your Hugging Face key (for a public endpoint, just press Enter): ", secret=True)
        api_key = api_key or "none"  # a public endpoint needs no key, but the client needs some text
        wait_until_awake(base_url, api_key)
        notes.append('Thinking: switch it with "reasoning_effort" in the extra_body of ChatOpenAI ("none" = off).')
        notes.append("Several answers per request (n > 1): the endpoint refuses them (HTTP 400).")

    elif provider == "ai-gateway":
        if IN_COLAB:
            notes.append("Colab runs outside the university network and usually cannot reach the AI-Gateway.")
        base_url = os.getenv("AI_GATEWAY_BASE") or "https://ai-gateway.uni-paderborn.de/v1"
        api_key = value("AI_GATEWAY_KEY", "Paste your AI-Gateway key: ", secret=True)
        if name not in GATEWAY_QWEN:
            notes.append(f"{name!r} is not one of the Qwen models of the course ({', '.join(GATEWAY_QWEN)}). "
                         "llm.root_client.models.list() lists the models of your account.")
        notes.append("Thinking cannot be switched with reasoning_effort here (the gateway ignores it): the model name decides. "
                     "gwdg.qwen3.8-27b-no-think = off, gwdg.qwen3.8-27b = on.")
        notes.append("Only in the university network or the VPN. About 10 requests per minute: if a cell pauses, the client waits and continues.")

    else:  # another OpenAI-compatible API
        if not name:
            raise ValueError("Give the model name of your provider as model_name.")
        base_url = value("OTHER_BASE", "Base URL of the API (e.g. https://api.openai.com/v1/): ")
        api_key = value("OTHER_KEY", "Your API key: ", secret=True)
        notes.append("Check in the documentation of your provider how thinking is switched; "
                     "the extra_body of ChatOpenAI may need other fields than reasoning_effort.")

    print(f"Provider: {provider} | model: {name}")  # the key is never printed
    print("Keep in mind:")
    for note in notes:
        print("  -", note)
    return base_url, api_key, name

**Before every `ChatOpenAI`, we call `setup_model()`** with the provider and the model, for example `setup_model({"model_name": "qwen3.5:9b", "provider": "huggingface"})`. It checks whether the combination exists, sets the address and the key, and prints what to keep in mind. It returns three values, which we split directly into `BASE_URL`, `API_KEY`, and `MODEL_NAME`. To switch the model, change the dict there.

### The embedding model runs on your computer

With every provider above, the **embedding model** runs on your computer: [all-MiniLM-L6-v2](https://huggingface.co/sentence-transformers/all-MiniLM-L6-v2), a small model that turns texts into vectors.

**Warning:** the first run downloads it from Hugging Face (about 90 MB). Without a GPU, it runs on the CPU, which is slower.

MiniLM reads at most **256 tokens** per text and silently cuts off the rest. So our chunks must be short.

In [ ]:
from huggingface_hub import try_to_load_from_cache

RETRIEVER_NAME = "sentence-transformers/all-MiniLM-L6-v2"  # embedding model: turns texts into vectors, runs on your computer
if not isinstance(try_to_load_from_cache(RETRIEVER_NAME, "config.json"), str):
    print(f"WARNING: {RETRIEVER_NAME} is downloaded on first use (about 90 MB). Without a GPU, it runs on the CPU, which is slower.")

## Loading the Documents

The Markdown files from 5.02, with the links removed.

In [ ]:
markdown_documents_path = "markdown_documents"

In [ ]:
def remove_markdown_links(text: str) -> str:
    # 1. Remove inline links: [text](url)
    text = re.sub(r'\[([^\]]*)\]\([^)]+\)', r'\1', text)

    # 2. Remove reference-style links: [text][ref]
    text = re.sub(r'\[([^\]]*)\]\[[^\]]*\]', r'\1', text)

    # 3. Remove malformed/broken bracket groups like [[][1][]]
    text = re.sub(r'\[+\s*\]+', '', text)

    # 4. Remove reference definitions like: [1]: https://...
    text = re.sub(r'^\s*\[[^\]]+\]:.*$', '', text, flags=re.MULTILINE)

    return text

In [ ]:
markdown_documents = sorted(os.listdir(markdown_documents_path))

md_files = []

for markdown_document in markdown_documents:
    markdown_document_path = os.path.join(markdown_documents_path, markdown_document)
    with open(markdown_document_path, encoding="utf-8") as file:
        md_files.append([markdown_document, remove_markdown_links(file.read())])

## The Chain from 5.03

### The vector store: build it once, then load it

Embedding all chunks takes a while on a CPU. So we do it only once:

* If you ran **5.03** in this folder, the index is saved in the folder `faiss_index_minilm`, and we just load it.
* Otherwise, we build it now and save it for the next notebooks.

In [ ]:
text_splitter = RecursiveCharacterTextSplitter.from_huggingface_tokenizer(AutoTokenizer.from_pretrained(RETRIEVER_NAME), chunk_size=250, chunk_overlap=25)
all_splits = text_splitter.create_documents(
    texts=[x[1] for x in md_files],
    metadatas=[{"source": x[0]} for x in md_files],
)

retriever_model = HuggingFaceEmbeddings(model_name=RETRIEVER_NAME)

if os.path.exists("faiss_index_minilm"):
    db = FAISS.load_local("faiss_index_minilm", retriever_model, allow_dangerous_deserialization=True)  # our own file, so loading is safe
else:
    db = FAISS.from_documents(all_splits, embedding=retriever_model)
    db.save_local("faiss_index_minilm")
print(db.index.ntotal, "chunks in the vector store")

For reranking, the retriever returns the **10** most similar chunks:

* **Without reranking**, we simply use the first 3 of them.
* **With reranking**, a second model reads all 10 again and picks the best 3.

In [ ]:
retriever = db.as_retriever(search_kwargs={"k": 10})

In [ ]:
BASE_URL, API_KEY, MODEL_NAME = setup_model({"model_name": "qwen3.5:9b", "provider": "huggingface"})  # provider and model: see the table in the setup
llm = ChatOpenAI(
    model=MODEL_NAME,  # on the AI-Gateway, the model decides about thinking: gwdg.qwen3.8-27b-no-think (off) or gwdg.qwen3.8-27b (on)
    base_url=BASE_URL,
    api_key=API_KEY,
    temperature=0,
    extra_body={
        "max_tokens": 512,  # the token limit (here, because Ollama ignores max_completion_tokens, the name that ChatOpenAI uses)
        "reasoning_effort": "none",  # thinking off; "low", "medium", or "high" switches it on (Ollama, Hugging Face endpoint)
    },
)

In [ ]:
prompt_template = """
You are an assistant for question-answering tasks. Use the following pieces of retrieved context to answer the question. If you don't know the answer, just say that you don't know. Use three sentences maximum and keep the answer concise.
Question: {question}
Context: {context}
Answer:
"""
prompt = PromptTemplate.from_template(prompt_template)
print(prompt.template)

## Part 1: The Cross-Encoder Reranker

**In short: a second model reads the question and each chunk together, and sorts the chunks again.**

![](https://raw.githubusercontent.com/UKPLab/sentence-transformers/master/docs/img/Bi_vs_Cross-Encoder.png)

Source: ([SBERT.net (2024)](https://www.sbert.net/examples/applications/cross-encoder/README.html))

| | Bi-encoder (our embedding model) | Cross-encoder (the reranker) |
|---|---|---|
| Reads | question and chunk **separately** | question and chunk **together** |
| Returns | one vector per text | one score between 0 and 1 per pair |
| Speed | fast: the chunk vectors are computed once | slow: one model run per pair |
| Good for | searching millions of chunks | sorting the 10 retrieved chunks again |

([Reimers and Gurevych, 2019](https://doi.org/10.48550/arXiv.1908.10084))

The rerankers in this notebook are small models that run on your own computer. **Warning:** the first run downloads them (the BGE reranker is about 1.1 GB, FlashRank is much smaller). Without a GPU, they run on the CPU, which is slower.

In [ ]:
model = HuggingFaceCrossEncoder(model_name="BAAI/bge-reranker-base")
compressor = CrossEncoderReranker(model=model, top_n=3)
compression_retriever = ContextualCompressionRetriever(
    base_compressor=compressor, base_retriever=retriever
)

Which chunks come first, without and with reranking? The numbers are the ids of the chunks:

In [ ]:
print([x.id for x in retriever.invoke("Who plays Daenerys Targaryen?")[:3]])
print([x.id for x in compression_retriever.invoke("Who plays Daenerys Targaryen?")])

### Your turn

Complete the `"compression"` branch in `invoke_reranking_rag_chain()`: build the `ContextualCompressionRetriever` from above, and use it to retrieve the chunks.

Then run both cells below and compare: which chunks does the model get, and does the answer change?

In [ ]:
def invoke_reranking_rag_chain(question, retriever_type="basic"):
    if retriever_type == "basic":
        retrieved_docs = retriever.invoke(question)[:3]  # the 3 most similar chunks, without reranking
    elif retriever_type == "compression":
        # TODO: implement compression retriever in here
        pass

    elif retriever_type == "flash":
        # TODO: Implement Flash retriever

        # your code here
        pass

    else:
        raise NotImplementedError("No other retrievers implemented yet.")
    for doc in retrieved_docs:
        display(Markdown(f"### {doc.metadata['source']}"))
        display(Markdown(doc.page_content))
    print("#" * 50)
    input_prompt = prompt.invoke({"question": question, "context": "\n\n".join(doc.page_content for doc in retrieved_docs)})
    answer = llm.invoke(input_prompt).content
    return answer

In [ ]:
question = "Who plays Daenerys Targaryen?"
answer = invoke_reranking_rag_chain(question, "basic")
print("Answer:", answer)

In [ ]:
question = "Who plays Daenerys Targaryen?"
answer = invoke_reranking_rag_chain(question, "compression")
print("Answer:", answer)

## Part 2: FlashRank

**In short: the same idea with a smaller and faster reranker.**

[FlashRank](https://github.com/PrithivirajDamodaran/FlashRank) uses tiny cross-encoders that run fast on a normal CPU. LangChain wraps it as `FlashrankRerank`.

### Your turn

1. Build a `FlashrankRerank(top_n=3)` compressor and a `ContextualCompressionRetriever` with it, like for the cross-encoder.
2. Add it as the `"flash"` branch of the function below.
3. Compare the chunks and the answer with the other two retrievers.

In [ ]:
# TODO: Implement Flash retriever

# your code here

In [ ]:
def invoke_reranking_rag_chain(question, retriever_type="basic"):
    if retriever_type == "basic":
        retrieved_docs = retriever.invoke(question)[:3]  # the 3 most similar chunks, without reranking
    elif retriever_type == "compression":
        # TODO: implement compression retriever in here
        pass
    elif retriever_type == "flash":
        # TODO: Implement Flash retriever

        # your code here
        pass

    else:
        raise NotImplementedError("No other retrievers implemented yet.")
    for doc in retrieved_docs:
        display(Markdown(f"### {doc.metadata['source']}"))
        display(Markdown(doc.page_content))
    print("#" * 50)
    input_prompt = prompt.invoke({"question": question, "context": "\n\n".join(doc.page_content for doc in retrieved_docs)})
    answer = llm.invoke(input_prompt).content
    return answer

In [ ]:
question = "Who plays Daenerys Targaryen?"
answer = invoke_reranking_rag_chain(question, "flash")
print("Answer:", answer)

## What You Learned

* The retriever returns more chunks than we need (here 10). A **reranker** sorts them again and keeps the best 3.
* A **cross-encoder** reads the question and the chunk together. That makes it more precise than the embedding model, but too slow for searching everything.
* In LangChain, a `ContextualCompressionRetriever` puts the reranker behind the normal retriever.

Next: **5.05 RAG Query Rewriting**. A better question before the search.